# ARC-AGI-3 — Chronos v14 PLM (Puzzle Language Model) submission

**Setup (one-time):** upload the `v14` folder contents as a private Kaggle
dataset named **`v14-plm`** containing:
- `plm/` (the whole package)
- `my_agent.py`
- `plm_weights.pt` (from train_wm.py; OPTIONAL — without it the agent
  runs its bandit fallback tier)

Attach that dataset + the competition data to this notebook.
Unlike v13 there is no giant `%%writefile` cell: the PLM is multi-file,
so code ships via the dataset and gets staged into `/kaggle/working`.

Integrity note: no solution caches, no engine sources. The PLM weights
are a general dynamics prior trained offline on the PUBLIC games; all
adaptation on the hidden eval happens in-episode.

In [ ]:
# Competition environment wheels (torch is preinstalled on Kaggle)
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
# Stage the PLM code + weights from the attached dataset, then sanity-check.
import os
!cp -r /kaggle/input/v14-plm/plm /kaggle/working/plm
!cp /kaggle/input/v14-plm/my_agent.py /kaggle/working/my_agent.py
!cp /kaggle/input/v14-plm/plm_weights.pt /kaggle/working/plm_weights.pt 2>/dev/null || echo 'NOTE: no weights in dataset - bandit fallback will carry the run'

# paste-mangling / truncation guard (catches bad copies in 2 seconds)
import ast
for f in ['/kaggle/working/my_agent.py'] + \
         [f'/kaggle/working/plm/{m}' for m in os.listdir('/kaggle/working/plm') if m.endswith('.py')]:
    ast.parse(open(f).read()); print('syntax OK:', f)

# full module smoke test (interactive runs only - skip during scoring rerun)
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !cd /kaggle/working && python -m plm.smoke

In [ ]:
import os
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # wait for the evaluation gateway, then stage the official harness
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    # agent file + the plm package + weights, all side by side so
    # my_agent.py's sys.path.insert(dirname(__file__)) finds the package
    !cp /kaggle/working/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    !cp -r /kaggle/working/plm /kaggle/working/ARC-AGI-3-Agents/agents/templates/plm
    !cp /kaggle/working/plm_weights.pt /kaggle/working/ARC-AGI-3-Agents/agents/templates/plm_weights.pt 2>/dev/null || true
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py','w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
""")
    with open('/kaggle/working/ARC-AGI-3-Agents/.env','w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
""")
    # PYTHONUNBUFFERED -> real-time Logs tab; tee -> downloadable artifact
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg PYTHONUNBUFFERED=1 python main.py --agent myagent 2>&1 | tee /kaggle/working/v14_run.log

this only runs if you submit to the competition, not when you do tests


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(data=[['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score'])
    submission.to_parquet('/kaggle/working/submission.parquet',index=False)

This is a dummy submission fallback, important to keep
